In [1]:
!nvidia-smi

Thu Sep 10 10:47:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
import sys, torch
print("Python:", sys.version.split()[0])
print("Torch :", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU   :", torch.cuda.get_device_name(0))

Python: 3.11.13
Torch : 2.6.0+cu124 | CUDA: True
GPU   : NVIDIA A100-SXM4-40GB


In [3]:
import os
REPO_DIR = "/content/silent_speech"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/MatteoFasulo/silent_speech.git {REPO_DIR}

!sed -i '/norm_layer=norm_layer,/d' {REPO_DIR}/architecture.py
print("remaining 'norm_layer=norm_layer,' lines (want 0):")
!grep -c "norm_layer=norm_layer," {REPO_DIR}/architecture.py || echo 0
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

remaining 'norm_layer=norm_layer,' lines (want 0):
0
0
cwd: /content/silent_speech


In [4]:
!pip install -q onnx onnxruntime timm jiwer unidecode omegaconf praat-textgrids \
    librosa speechbrain torchinfo torchprofile h5py transformers flashlight-text

from torchaudio.models.decoder import ctc_decoder
print("torchaudio ctc_decoder import OK")

torchaudio ctc_decoder import OK


In [5]:
from google.colab import drive
drive.mount("/content/drive")

import os, shutil
# ======================= EDIT THESE =======================
DRIVE_PROJECT = "/content/drive/MyDrive/silent_speech"                   # your project folder
MODEL_CKPT    = f"{DRIVE_PROJECT}/output/model_20260831_044634_best.pt"  # your 40.83% checkpoint
DATA_PATH     = DRIVE_PROJECT            # so $DATA_PATH/Gaddy/h5/emg_dataset.h5 is your h5
KENLM_SRC     = f"{DRIVE_PROJECT}/KenLM" # folder with lm.bin + gaddy_lexicon.txt
# ==========================================================

REPO_DIR = "/content/silent_speech"
os.chdir(REPO_DIR)
os.environ["DATA_PATH"] = DATA_PATH

dst = os.path.join(REPO_DIR, "KenLM")
if os.path.exists(KENLM_SRC) and not os.path.exists(dst):
    shutil.copytree(KENLM_SRC, dst)

h5 = os.path.expandvars("$DATA_PATH/emg_dataset.h5")
for label, p in [("h5 dataset", h5), ("checkpoint", MODEL_CKPT),
                 ("KenLM lm.bin", os.path.join(dst, "lm.bin")),
                 ("KenLM lexicon", os.path.join(dst, "gaddy_lexicon.txt"))]:
    print(("OK       " if os.path.exists(p) else "MISSING  ") + f"{label}: {p}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OK       h5 dataset: /content/drive/MyDrive/silent_speech/emg_dataset.h5
OK       checkpoint: /content/drive/MyDrive/silent_speech/output/model_20260831_044634_best.pt
OK       KenLM lm.bin: /content/silent_speech/KenLM/lm.bin
OK       KenLM lexicon: /content/silent_speech/KenLM/gaddy_lexicon.txt


In [6]:
import os, glob, json

# find your dataset file (prefer Drive)
cands = sorted(set(glob.glob("/content/drive/MyDrive/**/emg_dataset.h5", recursive=True)))
if not cands:
    cands = sorted(set(glob.glob("/content/**/emg_dataset.h5", recursive=True)))
print("candidates:")
for c in cands:
    print("   ", c, f"({os.path.getsize(c)/1e6:.0f} MB)")
assert cands, "No emg_dataset.h5 found — run  !find /content/drive -iname '*.h5'  and tell me the path."
H5_REAL = cands[0]
print("\nusing h5:", H5_REAL)

# write the ABSOLUTE path into both repo configs (repo reads h5_path from these JSONs)
for cfg in ["config/recognition_model.json", "config/transduction_model.json"]:
    p = os.path.join("/content/silent_speech", cfg)
    if os.path.exists(p):
        d = json.load(open(p))
        d["h5_path"] = H5_REAL
        json.dump(d, open(p, "w"), indent=4)
        print("patched", cfg)

print("exists:", os.path.exists(H5_REAL))

candidates:
    /content/drive/MyDrive/silent_speech/emg_dataset.h5 (6866 MB)

using h5: /content/drive/MyDrive/silent_speech/emg_dataset.h5
patched config/recognition_model.json
patched config/transduction_model.json
exists: True


In [7]:
!cd /content/silent_speech && DATA_PATH="{DATA_PATH}" python recognition_model.py --evaluate_saved "{MODEL_CKPT}"

2026-09-10 10:47:33.709685: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-10 10:47:33.733635: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1789037253.757481    5078 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1789037253.764608    5078 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-09-10 10:47:33.785977: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [8]:
import os, torch, numpy as np, torch.nn.functional as F
os.chdir("/content/silent_speech")
from data_utils import load_config
from hdf5_dataset import H5EmgDataset
from architecture import EMGTransformer

FLAGS  = load_config(os.path.join("config", "recognition_model.json"))
device = "cuda" if torch.cuda.is_available() else "cpu"

testset = H5EmgDataset(dev=False, test=True)          # silent test set
n_chars = len(testset.text_transform.chars)
print("test examples:", len(testset), "| num_outs:", n_chars + 1)

model = EMGTransformer(
    num_features=testset.num_features, num_outs=n_chars + 1,
    in_chans=FLAGS.in_chans, embed_dim=FLAGS.embed_dim,
    n_layer=FLAGS.num_layers, n_head=FLAGS.num_heads, mlp_ratio=FLAGS.mlp_ratio,
    attn_drop=FLAGS.dropout, proj_drop=FLAGS.dropout, freeze_blocks=FLAGS.freeze_blocks,
).to(device)
model.load_state_dict(torch.load(MODEL_CKPT, map_location=device), strict=True)
model.eval()
print("params:", sum(p.numel() for p in model.parameters()))

class Wrap(torch.nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): return self.m(x, x, x)      # only raw EMG used; returns raw logits
wrap = Wrap(model).eval().to(device)

ONNX_FP32 = "/content/model_fp32.onnx"
example = testset[0]["raw_emg"].unsqueeze(0).to(device)     # (1, T, 8)
torch.onnx.export(
    wrap, example, ONNX_FP32,
    input_names=["emg"], output_names=["logits"],
    dynamic_axes={"emg": {0: "batch", 1: "time"}, "logits": {0: "batch", 1: "time"}},
    opset_version=17, dynamo=False)
print("exported:", ONNX_FP32)

import onnxruntime as ort
sess = ort.InferenceSession(ONNX_FP32, providers=["CPUExecutionProvider"])
maxdiff = 0.0
for i in range(5):
    x = testset[i]["raw_emg"].unsqueeze(0)
    with torch.no_grad():
        yt = wrap(x.to(device)).cpu().numpy()
    yo = sess.run(None, {"emg": x.numpy().astype(np.float32)})[0]
    maxdiff = max(maxdiff, float(np.abs(yt - yo).max()))
print(f"torch vs ONNX max abs diff: {maxdiff:.2e}  (want < 1e-3)")

test examples: 99 | num_outs: 38
params: 2614694


/content/silent_speech/architecture.py:260: TracerWarning: Converting a tensor to a Python float might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  scale_factor = 1 / math.sqrt(q.size(-1))
/content/silent_speech/architecture.py:100: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  pad_length = max(length - self.max_relative_pos, 0)
/content/silent_speech/architecture.py:101: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not genera

exported: /content/model_fp32.onnx
torch vs ONNX max abs diff: 3.14e-02  (want < 1e-3)


In [9]:
import os, numpy as np, onnxruntime as ort
from onnxruntime.quantization import quantize_static, QuantType, QuantFormat, CalibrationDataReader

ONNX_INT8 = "/content/model_int8.onnx"
IN = ort.InferenceSession(ONNX_FP32, providers=["CPUExecutionProvider"]).get_inputs()[0].name

class EMGCalib(CalibrationDataReader):
    def __init__(self, dataset, n=40):
        self.items = []
        for i in range(min(n, len(dataset))):
            x = dataset[i]["raw_emg"].numpy().astype(np.float32)[None, ...]  # (1, T, 8)
            assert x.shape[-1] == 8, x.shape
            self.items.append({IN: x})
        print(f"calibration: {len(self.items)} real EMG examples")
        self.it = iter(self.items)
    def get_next(self): return next(self.it, None)

quantize_static(
    ONNX_FP32, ONNX_INT8, EMGCalib(testset, n=40),
    quant_format=QuantFormat.QDQ,
    weight_type=QuantType.QInt8, activation_type=QuantType.QInt8,
    per_channel=True)

mb = lambda p: os.path.getsize(p) / 1e6
print(f"FP32 {mb(ONNX_FP32):.2f} MB  ->  INT8 {mb(ONNX_INT8):.2f} MB  "
      f"({mb(ONNX_FP32)/mb(ONNX_INT8):.2f}x smaller)")

calibration: 40 real EMG examples


FP32 10.85 MB  ->  INT8 3.45 MB  (3.14x smaller)


In [10]:
import torch, numpy as np, torch.nn.functional as F, jiwer, tqdm
from torchaudio.models.decoder import ctc_decoder

def build_decoder(dset, beam_size=1500):
    tkns = [c for c in dset.text_transform.chars] + ["_"]
    return ctc_decoder(
        lexicon=os.path.join(FLAGS.lm_directory, "gaddy_lexicon.txt"),
        tokens=tkns, lm=os.path.join(FLAGS.lm_directory, "lm.bin"),
        blank_token="_", sil_token="|", nbest=1, lm_weight=2, beam_size=beam_size)

decoder = build_decoder(testset, beam_size=1500)

def onnx_wer(onnx_path, dset, decoder):
    sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
    name = sess.get_inputs()[0].name
    refs, preds = [], []
    for i in tqdm.tqdm(range(len(dset)), desc=os.path.basename(onnx_path)):
        ex = dset[i]
        x = ex["raw_emg"].numpy().astype(np.float32)[None, ...]
        y = sess.run(None, {name: x})[0]                 # (1, T', 38) raw logits
        logp = F.log_softmax(torch.from_numpy(y), dim=-1)
        beam = decoder(logp)
        pred = dset.text_transform.clean_text(" ".join(beam[0][0].words).strip())
        tgt  = dset.text_transform.clean_text(ex["text"])
        if tgt != "":
            refs.append(tgt); preds.append(pred)
    return jiwer.wer(refs, preds)

wer_fp32 = onnx_wer(ONNX_FP32, testset, decoder)
wer_int8 = onnx_wer(ONNX_INT8, testset, decoder)

mb = lambda p: os.path.getsize(p) / 1e6
margin = wer_fp32 * 1.10
print("\n================ RESULTS ================")
print(f"FP32 ONNX : WER {wer_fp32*100:6.2f}%   size {mb(ONNX_FP32):5.2f} MB")
print(f"INT8 ONNX : WER {wer_int8*100:6.2f}%   size {mb(ONNX_INT8):5.2f} MB")
print(f"Size reduction : {mb(ONNX_FP32)/mb(ONNX_INT8):.2f}x")
print(f"WER change     : {(wer_int8-wer_fp32)*100:+.2f} points")
print(f"RQ1 10% margin : INT8 WER <= {margin*100:.2f}%  ->  "
      f"{'PASS' if wer_int8 <= margin else 'FAIL'}")
print("=========================================")

model_int8.onnx: 100%|██████████| 99/99 [04:44<00:00,  2.87s/it]


================ RESULTS ================
FP32 ONNX : WER  40.83%   size 10.85 MB
INT8 ONNX : WER 100.00%   size  3.45 MB
Size reduction : 3.14x
WER change     : +59.17 points
RQ1 10% margin : INT8 WER <= 44.91%  ->  FAIL


In [11]:
import shutil, os
OUT = f"{DRIVE_PROJECT}/onnx"
os.makedirs(OUT, exist_ok=True)
for p in [ONNX_FP32, ONNX_INT8]:
    shutil.copy(p, OUT); print("saved", os.path.join(OUT, os.path.basename(p)))

saved /content/drive/MyDrive/silent_speech/onnx/model_fp32.onnx
saved /content/drive/MyDrive/silent_speech/onnx/model_int8.onnx


In [12]:
import os, onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType

ONNX_FP32 = "/content/model_fp32.onnx"
ONNX_DYN  = "/content/model_int8_dynamic.onnx"
quantize_dynamic(ONNX_FP32, ONNX_DYN, weight_type=QuantType.QInt8)   # weights-only

mb = lambda p: os.path.getsize(p)/1e6
print(f"FP32 {mb(ONNX_FP32):.2f} MB -> dynamic INT8 {mb(ONNX_DYN):.2f} MB ({mb(ONNX_FP32)/mb(ONNX_DYN):.2f}x)")

wer_dyn = onnx_wer(ONNX_DYN, testset, decoder)
print(f"dynamic INT8 WER: {wer_dyn*100:.2f}%   (RQ1 margin: <= 44.91%)")

  elem_type: 7
  shape {
    dim {
      dim_param: "unk__26"
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_value: 3
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_param: "unk__159"
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_param: "unk__201"
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_value: 3
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_param: "unk__334"
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_param: "unk__376"
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_value: 3
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_param: "unk__509"
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  sha

FP32 10.85 MB -> dynamic INT8 3.71 MB (2.93x)


model_int8_dynamic.onnx: 100%|██████████| 99/99 [03:49<00:00,  2.32s/it]

dynamic INT8 WER: 43.68%   (RQ1 margin: <= 44.91%)
